In [ ]:
!git clone https://github.com/Mr-Gump/seq-model.git
!pip install -r seq-model/requirements.txt
!pip install git+https://github.com/amphibian-dev/toad.git

In [1]:
import os
import sys

IS_COLAB = 'COLAB_RELEASE_TAG' in os.environ

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_DIR = '/content/drive/MyDrive/dataset/seq-data/'
    sys.path.append("/content/seq-model")
else:
    DATASET_DIR = './dataset'

In [2]:
import pandas as pd
import os

seq_data_path = os.path.join(DATASET_DIR, 'event_lst.csv')
sample_path = os.path.join(DATASET_DIR, 'tmp_0403.csv')

df = pd.read_csv(seq_data_path)
df['event_lst'] = df['event_lst'].apply(eval)
df['event_num'] = df['event_lst'].apply(len)

In [3]:
df_sample = pd.read_csv(sample_path)

In [4]:
df.drop_duplicates(subset=['sn'], inplace=True)
df_sample.drop_duplicates(subset=['sn'], inplace=True)

In [5]:
def get_event_seq(event_lst):
    return [event['event_id'] for event in event_lst]

def get_time_delta_seq(event_lst):
    return [event['time_delta'] for event in event_lst]

def get_value_seq(event_lst):
    return [
        (event['onway_amt'], event['onway_cnt'], event['borrow_amt'], event['is_same_pkg'])
        for event in event_lst
    ]

In [6]:
df['event_seq'] = df['event_lst'].apply(get_event_seq)
df['time_delta_seq'] = df['event_lst'].apply(get_time_delta_seq)
df['value_seq'] = df['event_lst'].apply(get_value_seq)

## 针对 log1p 压缩特性，将时间差序列转换为剩余时间序列，以更好地捕捉时间信息

In [7]:
import numpy as np

def handle_time_delta_seq(time_delta_seq):
    sum = 0
    total_sum = np.sum(time_delta_seq)
    for i in range(len(time_delta_seq)):
        sum += time_delta_seq[i]
        time_delta_seq[i] = total_sum - sum
    return time_delta_seq

df['time_delta_seq'] = df['time_delta_seq'].apply(handle_time_delta_seq)

In [8]:
df_merge = pd.merge(df_sample, df[['sn', 'event_seq','time_delta_seq','value_seq']], on='sn', how='inner')

In [9]:
df_merge

,sn,verify_time,target,reloan_score_2508_score,multi_borrow_count,global_borrow_count,event_seq,time_delta_seq,value_seq
0,CMS2026020910083479169892,1770606649,0,617.400262,3,2,"[1, 1, 5, 8, 1, 5, 8]","[3024168, 1191224, 1191134, 784507, 472485, 47...","[(0, 0, 0, 1), (0, 0, 0, 1), (0, 0, 4500.0, 1)..."
1,CMS2026022110565190425930,1771646230,0,629.405405,4,3,"[1, 1, 5, 8, 1, 5, 8, 1, 5, 7]","[57140470, 55307526, 55307436, 54900809, 54588...","[(0, 0, 0, 1), (0, 0, 0, 1), (0, 0, 4500.0, 1)..."
2,EDD20251130044636kYkJ36g7,1764496001,0,628.079043,1,3,"[1, 5, 1, 5, 8, 8, 1]","[871242, 871137, 859766, 859723, 443007, 44284...","[(0, 0, 0, 0), (0, 0, 3300.0, 0), (3300.0, 1, ..."
3,EDD20251130074911MNREI36Z,1764506956,1,580.394978,7,32,"[1, 1, 5, 8, 1, 5, 11, 10, 1, 1, 5, 1, 1, 1, 5...","[52501806, 44202414, 44202392, 43712039, 43369...","[(0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 2500.0, 0)..."
4,EDD20251130075400Jb6GcO7n,1764507245,1,604.912102,1,2,"[1, 5, 1, 5, 8, 8, 1]","[687597, 687539, 409833, 409772, 249179, 61, 0]","[(0, 0, 0, 1), (0, 0, 1800.0, 1), (1800.0, 1, ..."
...,...,...,...,...,...,...,...,...,...
98648,SVD20260301042307y5DvST74,1772356990,0,618.590732,34,34,"[1, 5, 5, 8, 8, 1, 5, 5, 8, 8, 1, 5, 7, 1, 5, ...","[14082970, 14082927, 14082892, 13675418, 13663...","[(0, 0, 0, 1), (0, 0, 3500.0, 1), (3500.0, 1, ..."
98649,SVD20260301064041HQ09RaY7,1772365244,0,598.078854,5,10,"[1, 5, 8, 1, 5, 4, 1, 5, 4, 4, 4, 11, 12, 9, 1...","[25860105, 25859939, 25426689, 24634594, 24634...","[(0, 0, 0, 0), (0, 0, 5000.0, 0), (5000.0, 1, ..."
98650,SVD2026030109320524EP5K5W,1772375529,0,590.785487,10,22,"[3, 5, 9, 1, 1, 5, 1, 5, 1, 5, 9, 9, 8, 1, 1, ...","[19042750, 19012081, 18484668, 18484406, 18435...","[(0, 0, 0, 0), (0, 0, 2700.0, 0), (2700.0, 1, ..."
98651,SVD20260301100941TW655kAf,1772334587,0,615.874396,13,64,"[1, 2, 2, 1, 1, 5, 8, 1, 5, 8, 1, 5, 8, 1, 5, ...","[59594476, 59510857, 59430440, 57005522, 56569...","[(0, 0, 0, 0), (0, 0, 0, 0), (0, 0, 0, 0), (0,..."


In [10]:
from sklearn.model_selection import train_test_split

df_merge = df_merge.sort_values('verify_time').reset_index(drop=True)

n = len(df_merge)
oot_start = int(n * 0.8)

df_in_time = df_merge.iloc[:oot_start]
df_oot = df_merge.iloc[oot_start:]

# 同一时间段内随机划分训练集(5/8)和测试集(3/8)，即总体 50%:30%
df_train, df_test = train_test_split(df_in_time, test_size=3/8, random_state=42, stratify=df_in_time['target'])

print(f"训练集: {len(df_train)} ({len(df_train)/n:.1%}), 日期范围: {df_train['verify_time'].min()} ~ {df_train['verify_time'].max()}")
print(f"测试集: {len(df_test)} ({len(df_test)/n:.1%}), 日期范围: {df_test['verify_time'].min()} ~ {df_test['verify_time'].max()}")
print(f"验证集(OOT): {len(df_oot)} ({len(df_oot)/n:.1%}), 日期范围: {df_oot['verify_time'].min()} ~ {df_oot['verify_time'].max()}")
print(f"\ntarget分布:")
print(f"  训练集: {df_train['target'].mean():.4f}")
print(f"  测试集: {df_test['target'].mean():.4f}")
print(f"  验证集: {df_oot['target'].mean():.4f}")

训练集: 49326 (50.0%), 日期范围: 1764496001 ~ 1771106923
测试集: 29596 (30.0%), 日期范围: 1764506956 ~ 1771106822
验证集(OOT): 19731 (20.0%), 日期范围: 1771107581 ~ 1772382556

target分布:
  训练集: 0.2165
  测试集: 0.2165
  验证集: 0.2357


In [11]:
from event_classifier_v2.main import main
import warnings

warnings.filterwarnings("ignore")

In [12]:
df_out = main(df_train, df_test, df_oot)

训练集 | 总量:49326  好:38647  坏:10679  坏率:21.65%  pos_weight:3.62
测试集 | 总量:29596   坏率:21.65%
OOT集  | 总量:19731    坏率:23.57%

模型参数量：35,129

Device: mps  |  pos_weight: 3.62

 Epoch │   TrLoss   TrAUC    TrKS │  ValLoss  ValAUC   ValKS   ValPR
──────────────────────────────────────────────────────────────────────
     1 │   1.0753  0.5774  0.1157 │   1.0397  0.6365  0.1988  0.3093 ◀ best
     2 │   1.0507  0.6215  0.1772 │   1.0361  0.6434  0.1992  0.3208 ◀ best
     3 │   1.0391  0.6397  0.2045 │   1.0204  0.6619  0.2297  0.3418 ◀ best
     4 │   1.0296  0.6522  0.2263 │   1.0303  0.6716  0.2473  0.3555 ◀ best
     5 │   1.0168  0.6680  0.2465 │   1.0160  0.6772  0.2532  0.3634 ◀ best
     6 │   1.0091  0.6746  0.2490 │   1.0093  0.6775  0.2505  0.3642
     7 │   1.0085  0.6767  0.2568 │   1.0063  0.6794  0.2572  0.3654 ◀ best
     8 │   1.0043  0.6807  0.2649 │   1.0074  0.6810  0.2615  0.3644 ◀ best
     9 │   1.0021  0.6821  0.2637 │   1.0023  0.6830  0.2683  0.3698 ◀ best
    10 │   0.999

KeyError: "['verify_dt'] not in index"

In [ ]:
df_test_res,df_oot_res = df_out[1],df_out[2]

In [ ]:
def prob_to_score(prob):
    factor = 20 / np.log(2)
    offset = 600 + factor * np.log(0.284)
    odds = prob / (1 - prob)
    score = offset - factor * np.log(odds)
    return score

In [ ]:
df_test_res['score'] = df_test_res['prob'].apply(prob_to_score)
df_oot_res['score'] = df_oot_res['prob'].apply(prob_to_score)

In [ ]:
import toad

bucket = toad.metrics.KS_bucket(df_oot_res['score'], df_oot_res['target'], bucket=10, method='quantile')
bucket

In [ ]:
bucket = toad.metrics.KS_bucket(df_oot_res['reloan_score_2508_score'], df_oot_res['target'], bucket=10, method='quantile')
bucket

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(df_test_res[['score', 'reloan_score_2508_score']].corr(), annot=True, cmap='RdBu_r', vmin=-1, vmax=1)
plt.show()

In [ ]:
# 1. 分别等频十分箱
df_oot_res['score_bin'] = pd.qcut(df_oot_res['score'], q=10, duplicates='drop')
df_oot_res['reloan_score_2508_score_bin'] = pd.qcut(df_oot_res['reloan_score_2508_score'], q=10, duplicates='drop')

# 2. 交叉统计 target 浓度（坏样本率）
cross = df_oot_res.pivot_table(
    values='target',
    index='score_bin',
    columns='reloan_score_2508_score_bin',
    aggfunc='mean'
)

# 3. 交叉统计每个格子的样本量（辅助参考）
cross_count = df_oot_res.pivot_table(
    values='target',
    index='score_bin',
    columns='reloan_score_2508_score_bin',
    aggfunc='count'
)

# 4. 热力图可视化
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.heatmap(cross, annot=True, fmt='.3f', cmap='YlOrRd', ax=axes[0])
axes[0].set_title('bad_rate')
axes[0].set_xlabel('reloan_score_2508_score_bin')
axes[0].set_ylabel('score_bin')

sns.heatmap(cross_count, annot=True, fmt='d', cmap='Blues', ax=axes[1])
axes[1].set_title('loan_cnt')
axes[1].set_xlabel('reloan_score_2508_score_bin')
axes[1].set_ylabel('score_bin')

plt.tight_layout()
plt.show()